# 01 · Data understanding and preprocessing

Explores the dataset, then turns the raw VisDrone-VID annotations into clean training and evaluation data shared by all models.

| Step | What it does |
|---|---|
| 0. EDA | Class balance, object sizes, aspect ratios, scene density, resolution — per split and per clip |
| 1. Frame step | Picks how many frames to skip, from how fast objects move between frames |
| 2. Label cleaning | Clips boxes to the image, drops degenerate boxes |
| 3. Ignored regions | Train: grey-fills them and drops boxes mostly inside. Eval: keeps them as `iscrowd=1` for every class |
| 4. Export | COCO JSON (torchvision + evaluator) and YOLO folders + `data.yaml` (Ultralytics) |
| 5. Augmentation | Defines the reproducible train-time augmentation and previews it |
| 6. Verification | Draws processed boxes on sample frames |

The train/val split follows the dataset folders: 56 `train/` clips for training, 7 `val/` clips as the holdout.

**Outputs** (in `OUT_ROOT`):
```
train/images/<clip>/<frame>.jpg      subsampled, ignored regions grey-filled
train/labels/<clip>/<frame>.txt      YOLO labels (class cx cy w h, normalised)
train/annotations_coco.json          COCO labels for torchvision training
val/images, val/labels               same processing, for Ultralytics' in-training metrics
val/gt_eval_coco.json                evaluation ground truth on the ORIGINAL val images
data.yaml                            Ultralytics config
preprocess_stats.json                counts for the report
```

## Setup

In [ ]:
import json
import random
import shutil
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# --- Paths ---
DATA_ROOT = Path("../multiple-object-video-detection-hw-2-ucu-ai-2026")  # unpacked dataset
OUT_ROOT = Path("../data_processed")
SPLITS = ["train", "val"]

# --- Frame subsampling ---
STEP = None                      # set an int to override the automatic choice
CANDIDATE_STEPS = [1, 2, 3, 5, 8, 10, 15, 20, 30]
IOU_TARGET = 0.5                 # pick the smallest step where same-object IoU drops to this
MAX_STEP = 10                    # cap so training keeps at least ~2.4k frames

# --- Cleaning and ignored regions ---
MIN_BOX_PX = 2                   # drop boxes thinner than this after clipping
IGNORE_OVERLAP = 0.5             # drop a training box if more than this fraction lies in an ignored region
FILL_VALUE = (114, 114, 114)     # grey, same as Ultralytics' padding colour
JPEG_QUALITY = 95

N_WORKERS = 8
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CLASSES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]
COLORS = [(0, 255, 0), (255, 255, 0), (0, 165, 255), (255, 0, 0),
          (0, 0, 255), (0, 255, 255), (128, 0, 255), (255, 128, 128)]  # BGR

## Load annotations

We use `annotations_coco.json` because it has `track_id` (needed for the frame-step analysis) and marks ignored regions with `iscrowd=1`.
Each split becomes three tables: `images`, `objects` (real objects) and `ignores` (ignored regions).

In [ ]:
def load_split(split):
    with open(DATA_ROOT / split / "annotations_coco.json") as f:
        coco = json.load(f)
    images = pd.DataFrame(coco["images"]).rename(columns={"id": "image_id"})
    anns = pd.DataFrame(coco["annotations"])
    anns[["x", "y", "w", "h"]] = pd.DataFrame(anns["bbox"].tolist(), index=anns.index).astype(float)
    anns = anns.drop(columns=["bbox", "segmentation"], errors="ignore")
    anns = anns.merge(images[["image_id", "video", "frame_id", "width", "height"]], on="image_id")
    objects = anns[anns["iscrowd"] == 0].reset_index(drop=True)
    ignores = anns[anns["iscrowd"] == 1].reset_index(drop=True)
    return images, objects, ignores

data = {s: load_split(s) for s in SPLITS}
pd.DataFrame({s: {"clips": im["video"].nunique(), "frames": len(im),
                  "objects": len(ob), "ignored regions": len(ig)}
              for s, (im, ob, ig) in data.items()})

## 0. Exploratory analysis

What we look at, and why it matters for later choices:
- **class balance** → loss weighting, rare-class augmentation, what per-class AP to expect;
- **object sizes** (absolute and after resizing) → input resolution, anchor sizes, P2 head;
- **box aspect ratios** → anchor aspect ratios;
- **objects per frame** (scene density) → `max_det`, number of DETR queries, NMS settings;
- **frame resolution and ignored area** → resizing policy, how much of each frame is masked.

Each is broken down by clip, because clips differ in altitude, camera angle and scene.

In [ ]:
# One table with train and val objects, plus a few derived columns
eda = pd.concat([data[s][1].assign(split=s) for s in SPLITS], ignore_index=True)
eda["cls"] = eda["category_id"].map(lambda c: CLASSES[c - 1])
eda["size"] = np.sqrt(eda.w * eda.h)                              # side of a square of the same area, px
eda["aspect"] = eda.w / eda.h
eda["size_1280"] = eda["size"] * 1280 / eda[["width", "height"]].max(axis=1)  # size after long side -> 1280
eda["coco_size"] = pd.cut(eda.w * eda.h, [0, 32**2, 96**2, np.inf], labels=["small", "medium", "large"])

### Class balance

In [ ]:
counts = eda.groupby(["cls", "split"]).size().unstack().reindex(CLASSES).fillna(0)
share = (counts / counts.sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
counts.plot.bar(ax=axes[0], logy=True, title="Objects per class (log scale)")
per_clip = (eda[eda.split == "train"].groupby(["video", "cls"]).size().unstack(fill_value=0)
            .reindex(columns=CLASSES, fill_value=0))
im = axes[1].imshow((per_clip.T / per_clip.sum(axis=1)).values, aspect="auto", cmap="viridis")
axes[1].set(yticks=range(len(CLASSES)), yticklabels=CLASSES, xlabel="train clip", title="Class share per clip")
plt.colorbar(im, ax=axes[1]); plt.tight_layout(); plt.show()
share

### Object sizes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
bins = np.logspace(0, np.log10(eda["size"].max()), 60)
for s in SPLITS:
    axes[0].hist(eda.loc[eda.split == s, "size"], bins=bins, alpha=0.6, density=True, label=s)
axes[0].axvline(32, color="k", ls="--", lw=1, label="COCO 'small' limit (32 px)")
axes[0].set(xscale="log", xlabel="object size √(w·h), px (original image)",
            ylabel="fraction of objects (density)", title="Object size"); axes[0].legend()

axes[1].hist(eda["size_1280"], bins=bins, alpha=0.7, color="C2")
for px, lbl in [(8, "stride 8 (P3)"), (4, "stride 4 (P2)")]:
    axes[1].axvline(px, ls=":", color="C3"); axes[1].text(px, axes[1].get_ylim()[1] * 0.9, lbl, rotation=90)
axes[1].set(xscale="log", xlabel="size after resizing long side to 1280 px",
            ylabel="number of objects", title="Size the model actually sees")

order = eda.groupby("cls")["size"].median().sort_values().index
axes[2].boxplot([eda.loc[eda.cls == c, "size"] for c in order], showfliers=False)
axes[2].set_xticks(range(1, len(order) + 1), order)  # works on old and new matplotlib
axes[2].set(yscale="log", ylabel="size, px", title="Size by class"); axes[2].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

print("COCO size buckets (% of objects):")
print((eda.groupby("split")["coco_size"].value_counts(normalize=True).unstack() * 100).round(1))
print("\nSize percentiles at 1280 px:", eda["size_1280"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_dict())

In [ ]:
# Per-clip median object size: shows how altitude changes scale between clips
clip_size = eda.groupby(["split", "video"])["size"].median().sort_values()
fig, ax = plt.subplots(figsize=(16, 3.5))
ax.bar(range(len(clip_size)), clip_size.values,
       color=["C1" if s == "val" else "C0" for s in clip_size.index.get_level_values(0)])
ax.set(xlabel="clip (sorted; orange = val)", ylabel="median object size, px", title="Median object size per clip")
plt.show()

### Box aspect ratios

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for c in CLASSES:
    ax.hist(np.log2(eda.loc[eda.cls == c, "aspect"]), bins=60, range=(-3, 3), histtype="step", label=c, density=True)
ax.set(xlabel="log2(width / height)   (0 = square, -1 = twice as tall)",
       ylabel="fraction of class objects (density)", title="Aspect ratio by class")
ax.legend(ncol=2); plt.show()
print("Aspect ratio (w/h) percentiles:", eda["aspect"].quantile([0.05, 0.25, 0.5, 0.75, 0.95]).round(2).to_dict())

### Scene density

In [ ]:
per_frame = pd.concat([
    data[s][0][["image_id", "video"]].assign(split=s)
    .merge(data[s][1].groupby("image_id").size().rename("n"), on="image_id", how="left").fillna({"n": 0})
    for s in SPLITS])

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for s in SPLITS:
    axes[0].hist(per_frame.loc[per_frame.split == s, "n"], bins=60, alpha=0.6, density=True, label=s)
axes[0].set(xlabel="objects per frame", ylabel="fraction of frames (density)", title="Objects per frame"); axes[0].legend()
clip_density = per_frame.groupby(["split", "video"])["n"].mean().sort_values()
axes[1].bar(range(len(clip_density)), clip_density.values,
            color=["C1" if s == "val" else "C0" for s in clip_density.index.get_level_values(0)])
axes[1].set(xlabel="clip (sorted; orange = val)", ylabel="mean objects per frame", title="Density per clip")
plt.tight_layout(); plt.show()
print("Objects per frame percentiles:", per_frame["n"].quantile([0.5, 0.9, 0.99, 1.0]).to_dict())

### Frame resolution and ignored area

In [ ]:
res = pd.concat([data[s][0].assign(split=s) for s in SPLITS]).groupby(["split", "video"])[["width", "height"]].first()
print("Train/val resolutions (clips):"); print(res.value_counts().rename("clips"))

# Test frames: if they are smaller than training frames, the same object covers fewer pixels at test time
test = pd.read_csv(DATA_ROOT / "test_images.csv").groupby("clip")[["width", "height"]].first()
print("\nTest resolutions (clips):"); print(test.value_counts().rename("clips"))
train_long = res.max(axis=1)
test_long = test.max(axis=1)

ign_area = pd.concat([data[s][2].assign(split=s) for s in SPLITS])
ign_area["frac"] = ign_area.w * ign_area.h / (ign_area.width * ign_area.height)
cover = ign_area.groupby(["split", "image_id"])["frac"].sum().clip(upper=1)   # overlaps make this an upper bound
print(f"\nFrames with ignored regions: {cover.groupby('split').size().to_dict()}")
print(f"Ignored area per such frame: median {cover.median():.1%}, 90th pct {cover.quantile(0.9):.1%}")

### Observations → decisions

Each check below informs a later choice; the cell prints the numbers to quote in the report.

| What we check | Decision it informs |
|---|---|
| Share of COCO-small objects, and of objects under 8 px after resizing to 1280 | Input resolution (1280 or higher), P2 head, SAHI tiling |
| Class shares | Per-class AP reporting; focal loss / rare-class augmentation experiments |
| Aspect-ratio percentiles | Anchor aspect ratios (refined with k-means for RetinaNet) |
| 99th percentile and max of objects per frame | `max_det` (submission cap is 500), number of DETR queries |
| Spread of per-clip object size and density | Scale augmentation; validating on whole held-out clips |
| Train/val vs test resolution | Whether objects in test frames are smaller in pixels than in training |
| Ignored area per frame | How much each frame loses to the grey fill (sections 3–4) |

In [ ]:
small = (eda["coco_size"] == "small").mean()
below8 = (eda["size_1280"] < 8).mean()
top2 = share["train"].sort_values(ascending=False).head(2)
rare = share["train"].sort_values().head(3)
asp = eda["aspect"].quantile([0.1, 0.5, 0.9]).round(2).tolist()
print(f"- {small:.0%} of objects are COCO-small (< 32x32 px); {below8:.0%} are under 8 px at 1280 px")
print(f"- Most common classes: {', '.join(f'{c} {v}%' for c, v in top2.items())}; "
      f"rarest: {', '.join(f'{c} {v}%' for c, v in rare.items())}")
print(f"- Aspect ratio w/h: 10th / 50th / 90th percentile = {asp}")
print(f"- Objects per frame: median {per_frame['n'].median():.0f}, 99th pct {per_frame['n'].quantile(0.99):.0f}, "
      f"max {per_frame['n'].max():.0f}")
print(f"- Median object size per clip ranges {clip_size.min():.0f}-{clip_size.max():.0f} px; "
      f"mean density per clip {clip_density.min():.0f}-{clip_density.max():.0f} objects")
print(f"- Long side: train/val median {train_long.median():.0f} px (max {train_long.max():.0f}), "
      f"test median {test_long.median():.0f} px (max {test_long.max():.0f})")
print(f"- Ignored area in frames that have it: median {cover.median():.0%}, 90th pct {cover.quantile(0.9):.0%}")

## 1. Choose the frame step

At 25 fps, neighbouring frames are near-duplicates. For each step *k* we match every object (by `track_id`) in frame *t* to itself in frame *t+k*
and compute the IoU of the two boxes. High IoU means the frames look the same; low IoU means objects moved or the camera panned, so the frame adds new information.

**Rule:** the smallest *k* where the median per-clip IoU drops to `IOU_TARGET`, capped at `MAX_STEP`.

In [ ]:
def same_track_iou(objects, k):
    """Mean IoU per clip between each object's box at frame t and at frame t+k."""
    a = objects[["video", "frame_id", "track_id", "x", "y", "w", "h"]]
    b = a.copy()
    b["frame_id"] -= k                     # row for frame t+k now sits at frame t
    m = a.merge(b, on=["video", "frame_id", "track_id"], suffixes=("", "_k"))
    ix = (np.minimum(m.x + m.w, m.x_k + m.w_k) - np.maximum(m.x, m.x_k)).clip(lower=0)
    iy = (np.minimum(m.y + m.h, m.y_k + m.h_k) - np.maximum(m.y, m.y_k)).clip(lower=0)
    inter = ix * iy
    m["iou"] = inter / (m.w * m.h + m.w_k * m.h_k - inter)
    return m.groupby("video")["iou"].mean()

train_objects = data["train"][1]
iou_table = pd.DataFrame({k: same_track_iou(train_objects, k) for k in CANDIDATE_STEPS}).T
median_iou = iou_table.median(axis=1)

auto_step = next((k for k in CANDIDATE_STEPS if median_iou[k] <= IOU_TARGET), CANDIDATE_STEPS[-1])
auto_step = min(auto_step, MAX_STEP)
STEP = STEP or auto_step

n_train_frames = len(data["train"][0])
print(pd.DataFrame({"median IoU": median_iou.round(3),
                    "train frames kept": [n_train_frames // k for k in CANDIDATE_STEPS]}))
print(f"\nChosen STEP = {STEP}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(iou_table.index, iou_table.values, color="grey", alpha=0.3, lw=1)
ax.plot(median_iou.index, median_iou.values, color="C0", lw=3, marker="o", label="median over clips")
ax.axhline(IOU_TARGET, color="C3", ls="--", label=f"target IoU = {IOU_TARGET}")
ax.axvline(STEP, color="C2", ls=":", label=f"chosen step = {STEP}")
ax.set(xlabel="frame step k", ylabel="same-object IoU between t and t+k",
       title="How quickly frames change (thin lines = individual clips)")
ax.legend()
plt.show()

## 2. Clean labels

Clip every box to the image borders, then drop boxes that became thinner than `MIN_BOX_PX` and any unexpected class id.

In [ ]:
def clean_boxes(objects):
    x1 = np.maximum(objects.x, 0)
    y1 = np.maximum(objects.y, 0)
    x2 = np.minimum(objects.x + objects.w, objects.width)
    y2 = np.minimum(objects.y + objects.h, objects.height)
    objects = objects.assign(x=x1, y=y1, w=x2 - x1, h=y2 - y1)
    keep = (objects.w >= MIN_BOX_PX) & (objects.h >= MIN_BOX_PX) & objects.category_id.between(1, 8)
    return objects[keep].reset_index(drop=True), int((~keep).sum())

stats = {}
for split in SPLITS:
    images, objects, ignores = data[split]
    cleaned, n_dropped = clean_boxes(objects)
    data[split] = (images, cleaned, ignores)
    stats[split] = {"frames_total": len(images), "boxes_raw": len(objects), "boxes_dropped_degenerate": n_dropped}
    print(f"{split}: dropped {n_dropped} degenerate boxes out of {len(objects)}")

## 3. Subsample frames and handle ignored regions

The same step is applied to train and val. Then, for the **training copies** of each frame:
- ignored regions are filled with grey, so the model never sees un-annotated objects labelled as background;
- object boxes whose area lies more than `IGNORE_OVERLAP` inside an ignored region are dropped, because their pixels are now grey.

The overlap is computed with an integral image of the ignore mask, so each box costs four lookups.

In [ ]:
def subsample(images, step):
    first = images.groupby("video")["frame_id"].transform("min")
    return images[(images["frame_id"] - first) % step == 0].reset_index(drop=True)

def ignore_fraction(objs, regions, W, H):
    """Fraction of each object's box area that lies inside any ignored region."""
    mask = np.zeros((H, W), np.uint8)
    for r in regions.itertuples():
        mask[int(r.y):int(np.ceil(r.y + r.h)), int(r.x):int(np.ceil(r.x + r.w))] = 1
    integral = cv2.integral(mask)  # shape (H+1, W+1)
    x1, y1 = objs.x.astype(int).values, objs.y.astype(int).values
    x2 = np.ceil(objs.x + objs.w).astype(int).clip(upper=W).values
    y2 = np.ceil(objs.y + objs.h).astype(int).clip(upper=H).values
    inside = integral[y2, x2] - integral[y1, x2] - integral[y2, x1] + integral[y1, x1]
    return inside / np.maximum((x2 - x1) * (y2 - y1), 1)

def drop_boxes_in_ignore(objects, ignores):
    keep = np.ones(len(objects), bool)
    ign_by_img = dict(tuple(ignores.groupby("image_id")))
    for img_id, idx in objects.groupby("image_id").groups.items():
        if img_id not in ign_by_img:
            continue
        objs = objects.loc[idx]
        frac = ignore_fraction(objs, ign_by_img[img_id], int(objs.width.iloc[0]), int(objs.height.iloc[0]))
        keep[idx] = frac <= IGNORE_OVERLAP
    return objects[keep].reset_index(drop=True), int((~keep).sum())

processed = {}
for split in SPLITS:
    images, objects, ignores = data[split]
    sel = subsample(images, STEP)
    objs = objects[objects.image_id.isin(sel.image_id)].reset_index(drop=True)
    ign = ignores[ignores.image_id.isin(sel.image_id)].reset_index(drop=True)
    train_objs, n_in_ignore = drop_boxes_in_ignore(objs, ign)
    processed[split] = dict(images=sel, objects_eval=objs, objects_train=train_objs, ignores=ign)
    stats[split].update(frames_kept=len(sel), boxes_kept_eval=len(objs),
                        boxes_dropped_in_ignore=n_in_ignore, boxes_kept_train=len(train_objs),
                        frames_with_ignore=int(sel.image_id.isin(ign.image_id).sum()))

pd.DataFrame(stats)

## 4. Export

**Images and YOLO labels** (both splits): frames with ignored regions are re-encoded with the grey fill; the others are copied unchanged.
Ultralytics finds each label by replacing `/images/` with `/labels/` in the image path.

**COCO JSON for training**: same images and boxes, plus the original ignored regions (`iscrowd=1`) in case a model wants to use them.

**Evaluation ground truth** (`val/gt_eval_coco.json`): points to the **original** val images, keeps every cleaned box (as the leaderboard does),
and adds each ignored region once **per class**. pycocotools only ignores a crowd region for detections of its own class,
and the dataset stores all ignored regions as class 1 (`person`), so without the copies a car detected inside a crowd area would count as a false positive.

In [ ]:
def export_images_and_yolo(split, p):
    out = OUT_ROOT / split
    ign_by_img = dict(tuple(p["ignores"].groupby("image_id")))
    obj_by_img = dict(tuple(p["objects_train"].groupby("image_id")))

    def work(row):
        src = DATA_ROOT / split / row.file_name
        dst = out / row.file_name
        dst.parent.mkdir(parents=True, exist_ok=True)
        regions = ign_by_img.get(row.image_id)
        if regions is None:
            shutil.copy2(src, dst)
        else:
            img = cv2.imread(str(src))
            for r in regions.itertuples():
                img[int(r.y):int(np.ceil(r.y + r.h)), int(r.x):int(np.ceil(r.x + r.w))] = FILL_VALUE
            cv2.imwrite(str(dst), img, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])

        label = out / "labels" / row.video / (Path(row.file_name).stem + ".txt")
        label.parent.mkdir(parents=True, exist_ok=True)
        objs = obj_by_img.get(row.image_id)
        lines = []
        if objs is not None:
            W, H = row.width, row.height
            for o in objs.itertuples():
                lines.append(f"{o.category_id - 1} {(o.x + o.w / 2) / W:.6f} {(o.y + o.h / 2) / H:.6f} "
                             f"{o.w / W:.6f} {o.h / H:.6f}")
        label.write_text("\n".join(lines))

    rows = list(p["images"].itertuples())
    with ThreadPoolExecutor(N_WORKERS) as pool:
        list(tqdm(pool.map(work, rows), total=len(rows), desc=f"{split} images"))

CATEGORIES = [{"id": i + 1, "name": n} for i, n in enumerate(CLASSES)]
IMAGE_COLS = ["image_id", "file_name", "width", "height", "video", "frame_id"]

def image_records(images):
    recs = images[IMAGE_COLS].rename(columns={"image_id": "id"}).to_dict("records")
    for r in recs:  # same key format as the submission's image_id: <clip>_<frame>
        r["key"] = f"{r['video']}_{int(r['frame_id']):07d}"
    return recs

def box_records(df, start_id, iscrowd, category_id=None):
    out = []
    for i, o in enumerate(df.itertuples()):
        out.append({"id": start_id + i, "image_id": int(o.image_id),
                    "category_id": int(category_id or o.category_id),
                    "bbox": [float(o.x), float(o.y), float(o.w), float(o.h)],
                    "area": float(o.w * o.h), "iscrowd": iscrowd})
    return out

def export_train_coco(split, p):
    objs = box_records(p["objects_train"], 1, 0)
    ign = box_records(p["ignores"], len(objs) + 1, 1)
    coco = {"images": image_records(p["images"]), "annotations": objs + ign, "categories": CATEGORIES}
    with open(OUT_ROOT / split / "annotations_coco.json", "w") as f:
        json.dump(coco, f)

def export_eval_gt(p, path):
    anns = box_records(p["objects_eval"], 1, 0)
    for c in range(1, len(CLASSES) + 1):  # one copy of every ignored region per class
        anns += box_records(p["ignores"], len(anns) + 1, 1, category_id=c)
    images = image_records(p["images"])
    for r in images:  # point to the original, un-filled val image
        r["file_name"] = str(Path("val") / r["file_name"])
    with open(path, "w") as f:
        json.dump({"images": images, "annotations": anns, "categories": CATEGORIES}, f)

for split in SPLITS:
    export_images_and_yolo(split, processed[split])
    export_train_coco(split, processed[split])
export_eval_gt(processed["val"], OUT_ROOT / "val" / "gt_eval_coco.json")

(OUT_ROOT / "data.yaml").write_text(
    f"path: {OUT_ROOT.resolve()}\ntrain: train/images\nval: val/images\n"
    f"nc: {len(CLASSES)}\nnames: {json.dumps(CLASSES)}\n")

stats["config"] = dict(step=int(STEP), iou_target=IOU_TARGET, min_box_px=MIN_BOX_PX,
                       ignore_overlap=IGNORE_OVERLAP, fill_value=FILL_VALUE, seed=SEED)
with open(OUT_ROOT / "preprocess_stats.json", "w") as f:
    json.dump(stats, f, indent=2)
print("Done:", sorted(p.name for p in OUT_ROOT.iterdir()))

## 5. Augmentation, resizing and normalisation

**Train-time augmentation** (applied on the fly in the training dataloader, never saved to disk). Every transform keeps tiny objects intact:
- horizontal flip: aerial scenes have no preferred left/right;
- mild scale/translate (0.8–1.2×): object size varies with drone altitude;
- brightness/contrast and hue/saturation: clips differ in lighting and time of day.

No vertical flip or large rotation: the camera is tilted, so "up" in the image is usually "far away".
Boxes that end up less than 30% visible or under 4 px² are dropped. The pipeline is seeded for reproducibility.

**Resizing and normalisation** happen inside each model, not here:
- torchvision detectors resize (`min_size`/`max_size`) and normalise with ImageNet mean/std in their built-in transform, so the dataloader only scales pixels to [0, 1];
- Ultralytics letterboxes to `imgsz` and scales to [0, 1] itself.

The target size (long side ≈ 1280 px) is set in each training notebook, so every model sees the same resolution.

In [ ]:
def build_train_aug(seed=SEED):
    return A.Compose(
        [
            A.HorizontalFlip(p=0.5),
            A.Affine(scale=(0.8, 1.2), translate_percent=(-0.05, 0.05), fill=FILL_VALUE, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
        ],
        bbox_params=A.BboxParams(format="coco", label_fields=["labels"], min_visibility=0.3, min_area=4),
        seed=seed,
    )

train_aug = build_train_aug()

## 6. Verify

In [ ]:
def draw(img, boxes, labels, thickness=1):
    img = img.copy()
    for (x, y, w, h), c in zip(boxes, labels):
        c = int(c)  # albumentations may return labels as floats
        p1, p2 = (int(x), int(y)), (int(x + w), int(y + h))
        cv2.rectangle(img, p1, p2, COLORS[c], thickness)
        cv2.putText(img, CLASSES[c], (p1[0], max(p1[1] - 3, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, COLORS[c], 1)
    return img

def load_processed(split, row):
    p = processed[split]
    img = cv2.imread(str(OUT_ROOT / split / row.file_name))
    objs = p["objects_train"][p["objects_train"].image_id == row.image_id]
    return img, objs[["x", "y", "w", "h"]].values.tolist(), (objs.category_id - 1).tolist()

# Pick a training frame that has ignored regions, so the grey fill is visible
imgs = processed["train"]["images"]
with_ign = imgs[imgs.image_id.isin(processed["train"]["ignores"].image_id)]
row = (with_ign if len(with_ign) else imgs).sample(1, random_state=SEED).iloc[0]

orig = cv2.imread(str(DATA_ROOT / "train" / row.file_name))
img, boxes, labels = load_processed("train", row)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, im, title in [(axes[0], orig, "original"),
                      (axes[1], draw(img, boxes, labels), "processed: grey fill + kept boxes")]:
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(title); ax.axis("off")
plt.suptitle(f"{row.video} frame {row.frame_id}")
plt.tight_layout(); plt.show()

In [ ]:
# Three augmented versions of the same frame
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax in axes:
    out = train_aug(image=img, bboxes=boxes, labels=labels)
    ax.imshow(cv2.cvtColor(draw(out["image"], out["bboxes"], out["labels"]), cv2.COLOR_BGR2RGB))
    ax.set_title(f"{len(out['bboxes'])} boxes"); ax.axis("off")
plt.suptitle("Train-time augmentation"); plt.tight_layout(); plt.show()

In [ ]:
# Round-trip check: YOLO label file -> pixel boxes must match the COCO boxes
label_file = OUT_ROOT / "train" / "labels" / row.video / (Path(row.file_name).stem + ".txt")
yolo = np.loadtxt(label_file, ndmin=2)
W, H = row.width, row.height
yolo_xywh = np.c_[(yolo[:, 1] - yolo[:, 3] / 2) * W, (yolo[:, 2] - yolo[:, 4] / 2) * H, yolo[:, 3] * W, yolo[:, 4] * H]
assert len(yolo) == len(boxes) and np.allclose(yolo_xywh, boxes, atol=0.01), "YOLO and COCO labels disagree"
print(f"OK: {len(boxes)} boxes match between YOLO and COCO exports")